Importing Required Libraries

The first step is importing the necessary Python libraries required for data processing, feature extraction, model training, and evaluation.

The scikit-learn library is used for machine learning algorithms, while pandas is used to load and manipulate the dataset.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import pandas as pd

Loading the Dataset

The dataset containing processed patent text is loaded using the pandas library. The dataset includes the cleaned textual content of patents along with their associated novelty tier labels.
This step helps verify that the dataset has been loaded correctly and allows us to inspect the structure of the data.

In [15]:

df = pd.read_csv("final_patent_dataset_balanced.csv")

print("Dataset loaded:", df.shape)

print(df.head())


Dataset loaded: (5000, 2)
                                          clean_text  novelty tier
0  methods systems providing proximity process re...             1
1  gateway anti theft security improved systems t...             2
2  managed shutdown distributed begins load balan...             2
3  semiconductor graphic wiring area semiconducto...             0
4  controlling virtualized functions receive proc...             0


Defining Features and Target Variable

The cleaned patent text is used as the input feature, while the novelty tier column serves as the target variable for the classification model.
Here:

clean_text contains the processed patent descriptions

novelty tier represents the classification label

In [16]:
X_text = df["clean_text"]

y = df["novelty tier"]


Converting Text into TF-IDF Features

Since machine learning models cannot directly process raw text, the textual data is transformed into numerical vectors using TF-IDF (Term Frequency – Inverse Document Frequency).

TF-IDF measures how important a word is within a document relative to the entire dataset.

Key parameters used in the vectorizer include limiting the vocabulary size, capturing multi-word phrases through n-grams, and filtering extremely rare or very common terms.

In [17]:
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True
)

X = vectorizer.fit_transform(X_text)

print("TF-IDF shape:", X.shape)


TF-IDF shape: (5000, 15000)


Feature Selection using Chi-Square Test

Text vectorization often produces a very large number of features. To reduce dimensionality and retain only the most relevant features, the Chi-Square statistical test is applied.

This technique selects features that have the strongest relationship with the target variable.

Feature selection improves training efficiency and can enhance model performance.

In [18]:
selector = SelectKBest(chi2, k=8000)

X = selector.fit_transform(X, y)

print("After feature selection:", X.shape)

After feature selection: (5000, 8000)


Splitting the Dataset into Training and Testing Sets

The dataset is split into training and testing subsets. The training set is used to train the models, while the testing set is used to evaluate their performance on unseen data.

Stratified sampling ensures that each class is proportionally represented in both training and testing datasets.

In [19]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (4000, 8000)
Test: (1000, 8000)


Defining Machine Learning Models

Several machine learning algorithms are used to determine which model performs best for the patent classification task.

The models used in this project include:

Multinomial Naive Bayes

Logistic Regression

Support Vector Machine

Random Forest

Gradient Boosting
Using multiple models allows a comparative evaluation to identify the most effective approach.

In [20]:
models = {

    "Multinomial Naive Bayes": MultinomialNB(),

    "Logistic Regression": LogisticRegression(max_iter=3000),

    "Support Vector Machine": LinearSVC(),

    "Random Forest": RandomForestClassifier(),

    "Gradient Boosting": GradientBoostingClassifier()

}


Hyperparameter Tuning

To improve model performance, hyperparameter tuning is performed using GridSearchCV. This technique tests multiple combinations of parameters using cross-validation to find the best configuration.

In [21]:
param_grids = {

    "Multinomial Naive Bayes": {
        "alpha": [0.1, 0.5, 1]
    },

    "Logistic Regression": {
        "C": [0.1, 1, 5]
    },

    "Support Vector Machine": {
        "C": [0.5, 1, 2]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 20]
    },

    "Gradient Boosting": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1]
    }

}

Model Training and Evaluation

Each model is trained using the training dataset, and the best parameters are selected using GridSearchCV. The trained model is then evaluated on the test dataset.

Model Performance Evaluation

The trained models are evaluated using several metrics:

Accuracy – Measures the overall correctness of predictions

Precision – Indicates how many predicted positives are correct

Recall – Measures how many actual positives are correctly identified

F1 Score – Harmonic mean of precision and recall

Confusion Matrix – Shows classification errors between classes

These evaluation metrics help determine which model performs best for patent novelty classification.

In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix

results = {}
best_models = {}

for name, model in models.items():

    grid = GridSearchCV(
        model,
        param_grids[name],
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    best_models[name] = best_model

    predictions = best_model.predict(X_test)

    results[name] = {
        "best_params": grid.best_params_,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
        "precision": precision_score(y_test, predictions, average="macro"),
        "recall": recall_score(y_test, predictions, average="macro"),
        "report": classification_report(y_test, predictions),
        "cm": confusion_matrix(y_test, predictions)
    }


In [23]:
name = "Multinomial Naive Bayes"
r = results[name]

print(name, "Results")
print("Accuracy   :", round(r["accuracy"],4))
print("Macro F1   :", round(r["macro_f1"],4))
print("Precision  :", round(r["precision"],4))
print("Recall     :", round(r["recall"],4))
print("Best Parameters:", r["best_params"])
print("\nClassification Report:")
print(r["report"])

print("Confusion Matrix:")
print(r["cm"])

Multinomial Naive Bayes Results
Accuracy   : 0.797
Macro F1   : 0.8
Precision  : 0.8105
Recall     : 0.797
Best Parameters: {'alpha': 0.1}

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.71      0.78       250
           1       0.65      0.75      0.70       250
           2       0.76      0.83      0.79       250
           3       0.97      0.90      0.93       250

    accuracy                           0.80      1000
   macro avg       0.81      0.80      0.80      1000
weighted avg       0.81      0.80      0.80      1000

Confusion Matrix:
[[177  58  15   0]
 [ 17 188  42   3]
 [  8  31 208   3]
 [  3  13  10 224]]


In [24]:
name = "Logistic Regression"
r = results[name]

print(name, "Results")
print("Accuracy   :", round(r["accuracy"],4))
print("Macro F1   :", round(r["macro_f1"],4))
print("Precision  :", round(r["precision"],4))
print("Recall     :", round(r["recall"],4))
print("Best Parameters:", r["best_params"])
print("\nClassification Report:")
print(r["report"])

print("Confusion Matrix:")
print(r["cm"])

Logistic Regression Results
Accuracy   : 0.905
Macro F1   : 0.9049
Precision  : 0.9051
Recall     : 0.905
Best Parameters: {'C': 5}

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.87       250
           1       0.86      0.84      0.85       250
           2       0.93      0.91      0.92       250
           3       0.98      0.99      0.98       250

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.90      1000
weighted avg       0.91      0.91      0.90      1000

Confusion Matrix:
[[220  25   4   1]
 [ 27 210  12   1]
 [ 11   7 228   4]
 [  0   2   1 247]]


In [25]:
name = "Support Vector Machine"
r = results[name]

print(name, "Results")
print("Accuracy   :", round(r["accuracy"],4))
print("Macro F1   :", round(r["macro_f1"],4))
print("Precision  :", round(r["precision"],4))
print("Recall     :", round(r["recall"],4))
print("Best Parameters:", r["best_params"])
print("\nClassification Report:")
print(r["report"])

print("Confusion Matrix:")
print(r["cm"])

Support Vector Machine Results
Accuracy   : 0.911
Macro F1   : 0.9107
Precision  : 0.9109
Recall     : 0.911
Best Parameters: {'C': 2}

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.89      0.87       250
           1       0.87      0.83      0.85       250
           2       0.93      0.94      0.94       250
           3       0.98      0.98      0.98       250

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000

Confusion Matrix:
[[222  26   2   0]
 [ 31 207  11   1]
 [  6   4 236   4]
 [  0   0   4 246]]


In [26]:
name = "Random Forest"
r = results[name]

print(name, "Results")
print("Accuracy   :", round(r["accuracy"],4))
print("Macro F1   :", round(r["macro_f1"],4))
print("Precision  :", round(r["precision"],4))
print("Recall     :", round(r["recall"],4))
print("Best Parameters:", r["best_params"])
print("\nClassification Report:")
print(r["report"])

print("Confusion Matrix:")
print(r["cm"])

Random Forest Results
Accuracy   : 0.89
Macro F1   : 0.8914
Precision  : 0.8934
Recall     : 0.89
Best Parameters: {'max_depth': None, 'n_estimators': 200}

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.82       250
           1       0.79      0.82      0.80       250
           2       0.98      0.93      0.95       250
           3       1.00      0.98      0.99       250

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000

Confusion Matrix:
[[209  40   0   1]
 [ 41 205   4   0]
 [  5  13 232   0]
 [  2   3   1 244]]


In [27]:
name = "Gradient Boosting"
r = results[name]

print(name, "Results")
print("Accuracy   :", round(r["accuracy"],4))
print("Macro F1   :", round(r["macro_f1"],4))
print("Precision  :", round(r["precision"],4))
print("Recall     :", round(r["recall"],4))
print("Best Parameters:", r["best_params"])
print("\nClassification Report:")
print(r["report"])

print("Confusion Matrix:")
print(r["cm"])

Gradient Boosting Results
Accuracy   : 0.921
Macro F1   : 0.92
Precision  : 0.9203
Recall     : 0.921
Best Parameters: {'learning_rate': 0.1, 'n_estimators': 200}

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.89       250
           1       0.89      0.81      0.85       250
           2       0.94      0.98      0.96       250
           3       0.98      1.00      0.99       250

    accuracy                           0.92      1000
   macro avg       0.92      0.92      0.92      1000
weighted avg       0.92      0.92      0.92      1000

Confusion Matrix:
[[226  23   1   0]
 [ 33 202  15   0]
 [  0   2 244   4]
 [  0   1   0 249]]
